# Museum Classifier — Semi-supervised: Decision Tree + Label Propagation / Self-Training

**Two preprocessing pipelines compared in parallel:**
- **Pipeline A** — ResNet18 (pretrained CNN feature extractor) → 512-d vectors
- **Pipeline B** — HOG + LBP + Color (classical descriptors) → ~1991-d vectors

**Semi-supervised strategy (per config):**
1. Split training images into labeled (small ratio) + unlabeled
2. Label Propagation / LabelSpreading / Self-Training assigns pseudo-labels to unlabeled
3. Decision Tree retrained on labeled + pseudo-labeled combined
4. Compare against labeled-only DT baseline — evaluated on the validation set

**Dataset structure on Google Drive:**
```
MyDrive/appliedAI/
├── training/
│   ├── museum-indoor/
│   └── museum-outdoor/
├── museum_validation/
│   ├── museum-indoor/
│   └── museum-outdoor/
└── test/
```

In [ ]:
# Install missing packages (run once)
!pip install -q scikit-image opencv-python-headless torch torchvision

## Environment Setup

Run this notebook **locally** or on **Google Colab** — the cell below detects the environment automatically.

- **Local**: uses `~/Documents/Prog/AppliedAI` as the base folder
- **Colab**: mounts Google Drive and uses `MyDrive/appliedAI` — upload your dataset there first

In [ ]:
import os

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/appliedAI')  # ← adjust Drive path if needed
else:
    BASE_DIR = Path.home() / 'Documents' / 'Prog' / 'AppliedAI'  # local path

print(f'Running in: {"Google Colab" if _in_colab() else "local"} environment')
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
import os, sys, time, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset

from sklearn.tree import DecisionTreeClassifier
from sklearn.semi_supervised import LabelPropagation, LabelSpreading, SelfTrainingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from skimage.feature import hog, local_binary_pattern
from skimage import color as skcolor
import cv2

%matplotlib inline
warnings.filterwarnings('ignore')

## Configuration

Adjust `BASE_DIR` to point to your dataset folder inside Google Drive if needed.

In [ ]:
TRAIN_DIR  = BASE_DIR / 'training'
VAL_DIR    = BASE_DIR / 'museum_validation'
TEST_DIR   = BASE_DIR / 'test'
OUTPUT_DIR = BASE_DIR / 'outputs_semisupervised'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['museum-indoor', 'museum-outdoor']   # must match folder names
IMG_SIZE     = 224
HC_SIZE      = (128, 128)
BATCH_SIZE   = 32
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42

print(f'Training dir    : {TRAIN_DIR}')
print(f'Validation dir  : {VAL_DIR}')
print(f'Test dir        : {TEST_DIR}')
print(f'Device          : {DEVICE}')

## Dataset Loader

In [ ]:
class MuseumDataset(Dataset):
    """Loads labeled images from a root folder containing one sub-folder per class."""
    def __init__(self, root: Path, classes: list, transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(classes):
            d = root / cls
            if not d.exists():
                print(f'[WARN] Folder not found: {d}'); continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
                for p in d.glob(ext):
                    self.samples.append((p, idx))
        counts = {cls: sum(1 for _, l in self.samples if l == i)
                  for i, cls in enumerate(classes)}
        print(f'  {root.name}: {len(self.samples)} images — ' +
              ', '.join(f'{cls}={n}' for cls, n in counts.items()))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

In [ ]:
print('Loading datasets...')
train_dataset = MuseumDataset(TRAIN_DIR, CLASSES)
val_dataset   = MuseumDataset(VAL_DIR,   CLASSES)

if len(train_dataset) == 0:
    raise RuntimeError('No training images found — check TRAIN_DIR.')
if len(val_dataset) == 0:
    raise RuntimeError('No validation images found — check VAL_DIR.')

## Step 1-A — Image Preprocessing: ResNet18 (512-d)

Pretrained ResNet18 with the classification head replaced by `nn.Identity()`.
Each image → 512-dimensional embedding. Run separately on training and validation sets.

In [ ]:
def extract_resnet_features(dataset: MuseumDataset) -> tuple:
    resnet_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    dataset.transform = resnet_tf

    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone.eval().to(DEVICE)

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    feats_l, labels_l = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            feats_l.append(backbone(imgs.to(DEVICE)).cpu().numpy())
            labels_l.append(lbls.numpy() if isinstance(lbls, torch.Tensor) else np.array(lbls))
    dataset.transform = None
    X = np.concatenate(feats_l)
    y = np.concatenate(labels_l)
    print(f'    shape={X.shape}')
    return X, y

print('[STEP 1-A] Extracting ResNet18 features...')
print('  Training set:')
X_resnet_train, y_train = extract_resnet_features(train_dataset)
print('  Validation set:')
X_resnet_val,   y_val   = extract_resnet_features(val_dataset)

## Step 1-B — Image Preprocessing: HOG + LBP + Color (~1991-d)

In [ ]:
def _hog_features(arr):
    gray = skcolor.rgb2gray(arr)
    f, _ = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                cells_per_block=(2, 2), visualize=True, feature_vector=True)
    return f

def _lbp_features(arr, n_bins=26):
    gray = (skcolor.rgb2gray(arr) * 255).astype(np.uint8)
    lbp  = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
    return hist.astype(float) / (hist.sum() + 1e-9)

def _color_features(arr, bins=32):
    feats = []
    for ch in range(3):
        h, _ = np.histogram(arr[:, :, ch], bins=bins, range=(0, 256))
        feats.extend(h / (h.sum() + 1e-9))
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    for ch in range(3):
        h, _ = np.histogram(hsv[:, :, ch], bins=bins, range=(0, 256))
        feats.extend(h / (h.sum() + 1e-9))
    for ch in range(3):
        c = arr[:, :, ch].astype(float)
        mu = c.mean(); sigma = c.std()
        skew = float(np.mean(((c - mu) / (sigma + 1e-9)) ** 3))
        feats.extend([mu / 255, sigma / 255, skew])
    return np.array(feats)

In [ ]:
def extract_handcrafted_features(dataset: MuseumDataset) -> tuple:
    feats_l, labels_l = [], []
    for img_path, label in dataset.samples:
        arr = np.array(Image.open(img_path).convert('RGB').resize(HC_SIZE, Image.BILINEAR))
        feats_l.append(np.concatenate([_hog_features(arr), _lbp_features(arr), _color_features(arr)]))
        labels_l.append(label)
    X = np.array(feats_l); y = np.array(labels_l)
    print(f'    shape={X.shape}  (HOG~1764 + LBP=26 + Color=201)')
    return X, y

def extract_handcrafted_single(img_path):
    arr = np.array(Image.open(img_path).convert('RGB').resize(HC_SIZE, Image.BILINEAR))
    return np.concatenate([_hog_features(arr), _lbp_features(arr), _color_features(arr)])

print('[STEP 1-B] Extracting HOG + LBP + Color features...')
print('  Training set:')
X_hand_train, _ = extract_handcrafted_features(train_dataset)
print('  Validation set:')
X_hand_val,   _ = extract_handcrafted_features(val_dataset)

## Semi-supervised Split Helper

Stratified split applied to the **training set only**: a given ratio per class receives real
labels; the rest are marked `-1` (unlabeled) for graph-based propagation methods.
The validation set is always fully labeled and used only for final evaluation.

In [ ]:
def make_semisup_split(X, y, labeled_ratio, rng_seed):
    """Stratified split: labeled_ratio% per class gets a real label, rest = -1."""
    rng = np.random.RandomState(rng_seed)
    labeled_idx = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        n_lab   = max(1, int(len(cls_idx) * labeled_ratio))
        labeled_idx.extend(rng.choice(cls_idx, n_lab, replace=False).tolist())
    labeled_idx   = np.array(labeled_idx)
    unlabeled_idx = np.setdiff1d(np.arange(len(y)), labeled_idx)
    y_semi = np.full(len(y), -1, dtype=int)
    y_semi[labeled_idx] = y[labeled_idx]
    return labeled_idx, unlabeled_idx, y_semi

## Step 2 — Semi-supervised Configurations (× 5)

Five configurations varying labeled ratio (10–30%), propagation method (LP / LS / Self-Training),
and Decision Tree depth.

In [ ]:
# (config_name, labeled_ratio, method, dt_max_depth, lp_kernel)
SEMISUP_CONFIGS = [
    ('LP_rbf_10pct_DT2',         0.10, 'LP',  2,    'rbf'),
    ('LP_knn_20pct_DT3',         0.20, 'LP',  3,    'knn'),
    ('LP_rbf_30pct_DT5',         0.30, 'LP',  5,    'rbf'),
    ('SelfTrain_20pct_DTfull',   0.20, 'ST',  None, None),
    ('LabelSpreading_30pct_DT4', 0.30, 'LS',  4,    'rbf'),
]

In [ ]:
def run_semisup_pipeline(X_train: np.ndarray, y_train: np.ndarray,
                          X_val:   np.ndarray, y_val:   np.ndarray,
                          pipe_label: str) -> tuple:
    """Scale → PCA (for LP/LS) → semi-sup propagation → DT retrain → eval on val."""
    scaler      = StandardScaler()
    X_train_sc  = scaler.fit_transform(X_train)
    X_val_sc    = scaler.transform(X_val)

    # PCA reduces cost of graph-based methods (O(N^2 x d))
    pca         = PCA(n_components=min(64, X_train_sc.shape[1]-1), random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_sc)
    print(f'  PCA 64-d: explained variance = {pca.explained_variance_ratio_.sum():.3f}')

    results = {}
    print(f'\n{"-"*64}')
    print(f'  {pipe_label}')
    print(f'  features={X_train.shape[1]}  train={len(X_train)}  val={len(X_val)}')
    print(f'{"-"*64}')

    for (name, labeled_ratio, method, dt_depth, lp_kernel) in SEMISUP_CONFIGS:
        print(f'\n  [CONFIG] {name}  (labeled={labeled_ratio:.0%}  method={method})')
        t0 = time.time()

        labeled_idx, unlabeled_idx, y_semi = make_semisup_split(
            X_train_pca, y_train, labeled_ratio, RANDOM_STATE)
        print(f'    Labeled={len(labeled_idx)}  Unlabeled={len(unlabeled_idx)}')

        # Semi-supervised propagation on PCA-reduced training features
        if method == 'LP':
            lp = LabelPropagation(kernel=lp_kernel, n_neighbors=7, max_iter=1000, n_jobs=-1)
            lp.fit(X_train_pca, y_semi)
            pseudo_all = lp.transduction_
        elif method == 'LS':
            ls = LabelSpreading(kernel=lp_kernel, alpha=0.2, max_iter=1000, n_jobs=-1)
            ls.fit(X_train_pca, y_semi)
            pseudo_all = ls.transduction_
        else:  # Self-Training
            base = DecisionTreeClassifier(max_depth=dt_depth, random_state=RANDOM_STATE)
            st   = SelfTrainingClassifier(base_estimator=base, threshold=0.75,
                                           criterion='threshold', max_iter=10, verbose=False)
            st.fit(X_train_pca, y_semi)
            pseudo_all = st.transduction_

        pseudo_unlab = pseudo_all[unlabeled_idx]
        pseudo_acc   = accuracy_score(y_train[unlabeled_idx], pseudo_unlab)
        print(f'    Pseudo-label accuracy: {pseudo_acc:.4f}')

        # Retrain DT on labeled + pseudo-labeled (full feature space, not PCA)
        X_combined = np.vstack([X_train_sc[labeled_idx], X_train_sc[unlabeled_idx]])
        y_combined = np.concatenate([y_train[labeled_idx], pseudo_unlab])
        dt = DecisionTreeClassifier(max_depth=dt_depth, min_samples_leaf=3,
                                     random_state=RANDOM_STATE)
        dt.fit(X_combined, y_combined)

        # Labeled-only baseline (evaluated on val)
        dt_base = DecisionTreeClassifier(max_depth=dt_depth, random_state=RANDOM_STATE)
        dt_base.fit(X_train_sc[labeled_idx], y_train[labeled_idx])
        f1_base = f1_score(y_val, dt_base.predict(X_val_sc), zero_division=0)

        # Evaluate semi-sup DT on validation set
        y_pred  = dt.predict(X_val_sc)
        y_proba = dt.predict_proba(X_val_sc)[:, 1]
        acc     = accuracy_score(y_val, y_pred)
        f1      = f1_score(y_val, y_pred, zero_division=0)
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        roc_auc = auc(fpr, tpr)
        elapsed = time.time() - t0

        results[name] = dict(
            dt=dt, scaler=scaler,
            acc=acc, f1=f1, f1_base=f1_base, roc_auc=roc_auc,
            fpr=fpr, tpr=tpr, cm=confusion_matrix(y_val, y_pred),
            y_pred=y_pred, y_proba=y_proba, y_test=y_val,
            pseudo_acc=pseudo_acc, n_lab=len(labeled_idx), n_unlab=len(unlabeled_idx),
            labeled_ratio=labeled_ratio, time=elapsed,
            report=classification_report(y_val, y_pred, target_names=CLASSES),
        )
        print(f'    Acc={acc:.4f}  F1={f1:.4f}  F1_base={f1_base:.4f}  '
              f'ΔF1={f1-f1_base:+.4f}  AUC={roc_auc:.4f}  t={elapsed:.1f}s')

    return results, scaler

In [ ]:
print('=' * 64)
print('  Running Pipeline A — ResNet18')
print('=' * 64)
res_r, sc_r = run_semisup_pipeline(X_resnet_train, y_train, X_resnet_val, y_val,
                                    'Pipeline A — ResNet18')

print('\n' + '=' * 64)
print('  Running Pipeline B — HOG + LBP + Color')
print('=' * 64)
res_h, sc_h = run_semisup_pipeline(X_hand_train, y_train, X_hand_val, y_val,
                                    'Pipeline B — HOG+LBP+Color')

all_results = {'resnet': res_r, 'handcrafted': res_h}
scalers     = {'resnet': sc_r,  'handcrafted': sc_h}

## Step 3 — Report

Comprehensive dashboard: accuracy/F1/AUC bars, F1 semi vs baseline, ΔF1 heatmap, ROC curves,
pseudo-label accuracy, best confusion matrices, labeled vs pseudo-labeled counts,
ratio→ΔF1 scatter, training times, cross-pipeline ΔF1, and full summary table.

In [ ]:
PIPE_META = {
    'resnet':      {'label': 'Pipeline A — ResNet18 (512-d)',        'color': '#4FC3F7'},
    'handcrafted': {'label': 'Pipeline B — HOG+LBP+Color (~1991-d)', 'color': '#81C784'},
}

def build_report(all_results: dict):
    cfg_names = [c[0] for c in SEMISUP_CONFIGS]
    n         = len(cfg_names)
    pipe_keys = list(PIPE_META.keys())
    palette   = sns.color_palette('Set2', n)

    fig = plt.figure(figsize=(26, 38))
    fig.patch.set_facecolor('#0d0f1a')
    gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.62, wspace=0.40)
    tkw  = dict(color='white', fontsize=10, fontweight='bold', pad=8)
    axbg = '#161929'

    def sa(ax):
        ax.set_facecolor(axbg); ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
        for sp in ax.spines.values(): sp.set_color('#2a2d3e')

    xlbls = [c.replace('_', '\n') for c in cfg_names]
    x, w  = np.arange(n), 0.38

    # Row 0: Accuracy / F1 / AUC grouped bars
    for col, (mkey, mtitle) in enumerate([('acc','Accuracy'),('f1','F1 Score (Semi)'),('roc_auc','ROC-AUC')]):
        ax = fig.add_subplot(gs[0, col]); sa(ax)
        for i, pk in enumerate(pipe_keys):
            vals = [all_results[pk][c][mkey] for c in cfg_names]
            bars = ax.bar(x + (i-0.5)*w, vals, w, color=PIPE_META[pk]['color'],
                          label=PIPE_META[pk]['label'], edgecolor='white', linewidth=0.4, alpha=0.88)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x()+bar.get_width()/2, v+0.012,
                        f'{v:.2f}', ha='center', color='white', fontsize=6)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6)
        ax.set_ylim(0, 1.15); ax.set_title(mtitle, **tkw)
        if col == 0: ax.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    # Row 1: F1 Semi vs Baseline + ΔF1 heatmap
    for col, pk in enumerate(pipe_keys):
        ax = fig.add_subplot(gs[1, col]); sa(ax)
        f1s  = [all_results[pk][c]['f1']      for c in cfg_names]
        f1bs = [all_results[pk][c]['f1_base'] for c in cfg_names]
        ax.bar(x - w/2, f1s,  w, color=PIPE_META[pk]['color'], label='Semi-sup DT',  alpha=0.90, edgecolor='white', lw=0.4)
        ax.bar(x + w/2, f1bs, w, color=PIPE_META[pk]['color'], label='Labeled-only', alpha=0.35, edgecolor='white', lw=0.4)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6)
        ax.set_ylim(0, 1.12)
        ax.set_title(f'F1: Semi vs Baseline\n{PIPE_META[pk]["label"]}', **tkw)
        ax.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    ax_heat = fig.add_subplot(gs[1, 2]); sa(ax_heat)
    delta_mat = np.array([[all_results[pk][c]['f1'] - all_results[pk][c]['f1_base']
                           for c in cfg_names] for pk in pipe_keys])
    lim = max(abs(delta_mat.min()), abs(delta_mat.max())) + 0.01
    im = ax_heat.imshow(delta_mat, cmap='RdYlGn', aspect='auto', vmin=-lim, vmax=lim)
    ax_heat.set_xticks(range(n)); ax_heat.set_xticklabels(xlbls, color='#ccc', fontsize=6)
    ax_heat.set_yticks([0,1]); ax_heat.set_yticklabels(['ResNet18','HOG+LBP+Color'], color='#ccc', fontsize=8)
    ax_heat.set_title('ΔF1 Heatmap (Semi − Baseline)\nGreen = semi helps', **tkw)
    for i in range(2):
        for j in range(n):
            ax_heat.text(j, i, f'{delta_mat[i,j]:+.3f}', ha='center', va='center',
                         color='black', fontsize=8, fontweight='bold')
    plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)

    # Row 2: ROC curves + pseudo-label accuracy
    for col, pk in enumerate(pipe_keys):
        ax = fig.add_subplot(gs[2, col]); sa(ax)
        ax.plot([0,1],[0,1],'w--',lw=1,alpha=0.35)
        for i, cn in enumerate(cfg_names):
            r = all_results[pk][cn]
            ax.plot(r['fpr'], r['tpr'], color=palette[i], lw=1.8,
                    label=f'{cn.split("_")[0]} ({r["roc_auc"]:.3f})')
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.set_title(f'ROC Curves\n{PIPE_META[pk]["label"]}', **tkw)
        ax.legend(fontsize=6, facecolor='#0d0f1a', labelcolor='white')

    ax_pseudo = fig.add_subplot(gs[2, 2]); sa(ax_pseudo)
    for i, pk in enumerate(pipe_keys):
        pa = [all_results[pk][c]['pseudo_acc'] for c in cfg_names]
        ax_pseudo.bar(x + (i-0.5)*w, pa, w, color=PIPE_META[pk]['color'],
                      label=PIPE_META[pk]['label'], edgecolor='white', linewidth=0.4, alpha=0.88)
    ax_pseudo.set_xticks(x); ax_pseudo.set_xticklabels(xlbls, color='#ccc', fontsize=6)
    ax_pseudo.set_ylim(0, 1.1)
    ax_pseudo.set_title('Pseudo-label Accuracy\n(quality of unlabeled propagation)', **tkw)
    ax_pseudo.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    # Row 3: Best confusion matrices + stacked labeled/pseudo counts
    for col, pk in enumerate(pipe_keys):
        best_cn = max(cfg_names, key=lambda c: all_results[pk][c]['f1'])
        ax = fig.add_subplot(gs[3, col]); sa(ax)
        cmap = 'Blues' if pk == 'resnet' else 'Greens'
        ConfusionMatrixDisplay(all_results[pk][best_cn]['cm'],
                               display_labels=CLASSES).plot(ax=ax, colorbar=False, cmap=cmap)
        ax.set_title(f'Best CM — {PIPE_META[pk]["label"]}\n{best_cn}', **tkw)
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')

    ax_stk = fig.add_subplot(gs[3, 2]); sa(ax_stk)
    pk_ref  = 'resnet'
    n_labs  = [all_results[pk_ref][c]['n_lab']   for c in cfg_names]
    n_unlabs= [all_results[pk_ref][c]['n_unlab'] for c in cfg_names]
    ax_stk.bar(range(n), n_labs,   color='#4CAF50', label='Labeled',        alpha=0.9)
    ax_stk.bar(range(n), n_unlabs, bottom=n_labs, color='#2196F3', label='Pseudo-labeled', alpha=0.65)
    ax_stk.set_xticks(range(n)); ax_stk.set_xticklabels(xlbls, color='#ccc', fontsize=6)
    ax_stk.set_title('Labeled vs Pseudo-labeled Counts\n(training set)', **tkw)
    ax_stk.legend(fontsize=8, facecolor='#0d0f1a', labelcolor='white')

    # Row 4: Labeled ratio vs ΔF1 scatter + time + cross-pipeline ΔF1
    ax_sc = fig.add_subplot(gs[4, 0]); sa(ax_sc)
    for i, pk in enumerate(pipe_keys):
        ratios = [all_results[pk][c]['labeled_ratio'] for c in cfg_names]
        deltas = [all_results[pk][c]['f1'] - all_results[pk][c]['f1_base'] for c in cfg_names]
        ax_sc.scatter(ratios, deltas, c=PIPE_META[pk]['color'], s=90,
                      label=PIPE_META[pk]['label'], edgecolors='white', linewidth=0.5, zorder=3)
    ax_sc.axhline(0, 'w', '--', lw=0.8, alpha=0.5)
    ax_sc.set_xlabel('Labeled Ratio'); ax_sc.set_ylabel('ΔF1')
    ax_sc.set_title('Labeled Ratio → ΔF1 Gain', **tkw)
    ax_sc.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    ax_time = fig.add_subplot(gs[4, 1]); sa(ax_time)
    for i, pk in enumerate(pipe_keys):
        times = [all_results[pk][c]['time'] for c in cfg_names]
        ax_time.bar(x + (i-0.5)*w, times, w, color=PIPE_META[pk]['color'],
                    label=PIPE_META[pk]['label'], edgecolor='white', linewidth=0.4, alpha=0.88)
    ax_time.set_xticks(x); ax_time.set_xticklabels(xlbls, color='#ccc', fontsize=6)
    ax_time.set_ylabel('seconds'); ax_time.set_title('Training Time (s)', **tkw)
    ax_time.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    ax_cross = fig.add_subplot(gs[4, 2]); sa(ax_cross)
    cross_d = [all_results['resnet'][c]['f1'] - all_results['handcrafted'][c]['f1']
               for c in cfg_names]
    bcolors = ['#4FC3F7' if d >= 0 else '#EF5350' for d in cross_d]
    ax_cross.bar(range(n), cross_d, color=bcolors, edgecolor='white', lw=0.5)
    ax_cross.axhline(0, 'w', '--', lw=0.8, alpha=0.5)
    ax_cross.set_xticks(range(n)); ax_cross.set_xticklabels(xlbls, color='#ccc', fontsize=6)
    ax_cross.set_title('ΔF1  (ResNet18 − HOG+LBP+Color)\nper semi-sup config', **tkw)
    for i, (bar, v) in enumerate(zip(ax_cross.patches, cross_d)):
        ax_cross.text(bar.get_x()+bar.get_width()/2,
                      v + (0.005 if v >= 0 else -0.016),
                      f'{v:+.3f}', ha='center', color='white', fontsize=7.5, fontweight='bold')

    # Row 5: Full summary table
    ax_tbl = fig.add_subplot(gs[5, :]); sa(ax_tbl); ax_tbl.axis('off')
    col_labels = ['Pipeline','Config','Labeled%','Accuracy','F1 Semi','F1 Base',
                  'ΔF1','Pseudo-Acc','AUC','Time(s)']
    rows = []
    for pk in pipe_keys:
        for cn in cfg_names:
            r = all_results[pk][cn]
            delta = r['f1'] - r['f1_base']
            rows.append([
                'A: ResNet18' if pk=='resnet' else 'B: HOG+LBP+Color',
                cn, f'{r["labeled_ratio"]:.0%}',
                f'{r["acc"]:.4f}', f'{r["f1"]:.4f}', f'{r["f1_base"]:.4f}',
                f'{delta:+.4f}', f'{r["pseudo_acc"]:.4f}',
                f'{r["roc_auc"]:.4f}', f'{r["time"]:.1f}',
            ])
    tbl = ax_tbl.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(7.5); tbl.scale(1, 1.80)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#2a2d3e')
        if row == 0:
            cell.set_facecolor('#1e2235'); cell.set_text_props(color='white', fontweight='bold')
        elif rows[row-1][0].startswith('A'):
            cell.set_facecolor('#0d1828' if row%2 else '#091220')
            if col == 6:
                try:
                    v = float(rows[row-1][6])
                    cell.set_text_props(color='#4CAF50' if v >= 0 else '#EF5350', fontweight='bold')
                except: cell.set_text_props(color='#B3E5FC')
            else: cell.set_text_props(color='#B3E5FC')
        else:
            cell.set_facecolor('#0d1a0f' if row%2 else '#09130b')
            if col == 6:
                try:
                    v = float(rows[row-1][6])
                    cell.set_text_props(color='#4CAF50' if v >= 0 else '#EF5350', fontweight='bold')
                except: cell.set_text_props(color='#C8E6C9')
            else: cell.set_text_props(color='#C8E6C9')
    ax_tbl.set_title('Full Comparison — Semi-supervised DT: Pipeline A vs Pipeline B', **tkw)

    fig.text(0.5, 0.988, 'Museum Classifier — Semi-supervised Decision Tree',
             ha='center', va='top', color='white', fontsize=17, fontweight='bold')
    fig.text(0.5, 0.975,
             'Step 3 Report  |  Pipeline A: ResNet18 (512-d)  vs  Pipeline B: HOG+LBP+Color (~1991-d)',
             ha='center', va='top', color='#aaa', fontsize=10)

    out = OUTPUT_DIR / 'report_semisupervised.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'\n[REPORT] Saved → {out}')
    return out

In [ ]:
report_path = build_report(all_results)

## Best Model Classification Reports

In [ ]:
for pk in all_results:
    best = max(all_results[pk], key=lambda c: all_results[pk][c]['f1'])
    print(f'[BEST — {PIPE_META[pk]["label"]}]  {best}')
    print(all_results[pk][best]['report'])
    print()

## Test Prediction

Uses the globally best model (across both pipelines and all 5 configs) to predict unlabeled
images in `TEST_DIR`. Saves a CSV with filename, predicted class, confidence, pipeline, and config.

In [ ]:
def predict_test(all_results: dict, scalers: dict):
    best_pk, best_cn, best_f1 = None, None, -1
    for pk in all_results:
        for cn in all_results[pk]:
            if all_results[pk][cn]['f1'] > best_f1:
                best_pk, best_cn, best_f1 = pk, cn, all_results[pk][cn]['f1']

    print(f'Best model: {PIPE_META[best_pk]["label"]} / {best_cn}  (F1={best_f1:.4f})')

    test_paths = []
    if TEST_DIR.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
            test_paths.extend(TEST_DIR.glob(ext))
    if not test_paths:
        print('[WARN] No test images found in TEST_DIR.'); return

    dt, scaler = all_results[best_pk][best_cn]['dt'], scalers[best_pk]

    if best_pk == 'resnet':
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Identity(); backbone.eval().to(DEVICE)
        tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        feats = []
        with torch.no_grad():
            for p in test_paths:
                img = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
                feats.append(backbone(img).cpu().numpy()[0])
    else:
        feats = [extract_handcrafted_single(p) for p in test_paths]

    X_test = scaler.transform(np.array(feats))
    preds  = dt.predict(X_test)
    probas = dt.predict_proba(X_test)[:, 1]

    out_csv = OUTPUT_DIR / 'predictions_semisupervised.csv'
    with open(out_csv, 'w') as f:
        f.write('filename,prediction,confidence_outdoor,pipeline,config\n')
        for p, pred, prob in zip(test_paths, preds, probas):
            f.write(f'{p.name},{CLASSES[pred]},{prob:.4f},{best_pk},{best_cn}\n')
    print(f'Predictions saved → {out_csv}')
    print(f'[DONE] All outputs in: {OUTPUT_DIR}')

predict_test(all_results, scalers)